* You can use as many cells as you need to answer the questions.
* Answer any questions in a markup cell.
* Plots should have labels on the axes, legends if appropriate etc.


In [23]:
import numpy as np

import matplotlib
import matplotlib.pyplot as plt

# Adjust default font size so that labels are readable in plots
font = {'size': 18}
matplotlib.rc('font', **font)
matplotlib.rcParams['savefig.dpi'] = 200
matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['figure.figsize'] = [6, 4]

# Q1

Suppose that there is an outbreak of a deadly disease - for example, perhaps avian flu has hopped to humans. Your doctor has just told you that you have tested positive for this disease. For people <mark>with the disease, the test is correct 90\% of the time</mark>. He also mentions that the disease is prevalent in the general population at a rate of <mark>0.1\%</mark> and that the <mark>false positive rate for the test is 1\%</mark>.

Should you worry? Explain your reasoning quantitatively.

Recall Bayes' Theorem 

\begin{equation*}
P(A|B) = \frac{P(B|A) P(A)}{P(B)}
\end{equation*}

and let $A = $ Actually sick and $B = $ Test positive. Such that $P(B|A) = 0.9$, $P(A) = 0.001$, and $P(B|A') = 0.01 $ (false positive)

By the law of total probability we get $P(B)$:
\begin{equation*}
P(B) = P (B|A) P(A) + P(B|A')P(A') = (0.9)(0.001) + (0.01)(1 - 0.001) = 0.01089
\end{equation*}

which we can then use to compute the probability that we are truly sick given that we tested positive.: 

\begin{equation*}
P(A|B) = \frac{P(B|A) P(A)}{P(B)} = \frac{(0.9)(0.001)}{(0.01089)} = 0.0826446281 \approx \boxed{8.3\%}
\end{equation*}

Despite the positive test, the probability of actually having the disease is only about $8.3\%$. Because the disease is rare to begin with, false positives outnumber true positives and the positive result therefore does not make infection likely. Although it raises the probability substantially from the initial $0.1\%$, which would *personally* make me worry.

# Q2

Experimental design: In the original HST guide star catalogue (created based on ground-based data), 60\% of the catalogue "objects" are actually binary stars. These are bad for guiding.

How large a sample of $N$ catalogue objects should be chosen to ensure that the probability that at least <mark>two sample objects are non-binary</mark> stars is at least <mark>99\%</mark>?

## a) 
Write down an analytic expression for $N$. Then solve it analytically or numerically.


We are given $P(\rm Binary) = 0.6$ and $P(\rm Non-Binary) = 0.4$, since we are looking for non-binary stars (indep trials that either succeed or fail) this is a binomial distribution with success probability $p = 0.4$ and failure probability $q = 0.6$. Looking for $N \geq 2$ at $99\%$ chance, and let $X$ be the number of NB stars: 
\begin{equation*}
P(X \geq 2) \geq 0.99
\end{equation*}
we can compute as the total probability ($1$) - the odds of getting $1$ and $0$ NB stars: 
\begin{equation*}
P(X \geq 2) = 1 - P(X=1) - P(X=0)
\end{equation*}
we can compute $P(X=1)$ and $P(X=0)$ with the binomial formula: 
\begin{equation*}
P(X = k) = \frac{N!}{k!(N-1)!}p^{k}q^{N-k} 
\end{equation*}
such that 
\begin{equation*}
P(X = 0) = \frac{N!}{0!(N-1)!}(0.4)^{0}(0.6)^{N-0} = (0.6)^N
\end{equation*}

\begin{equation*}
P(X = 1) = \frac{N!}{1!(N-1)!}(0.4)^{1}(0.6)^{N-1} = N(0.4)(0.6)^{N-1}
\end{equation*}

which gives the following critical condition for the probability of getting at least two NB being at least $99\%$:
\begin{equation*}
P(X \geq 2) = 1 - P(X=1) - P(X=0)
\end{equation*}

\begin{equation*}
\boxed{0.99 \geq 1 - N(0.4)(0.6)^{N-1} - (0.6)^N}
\end{equation*}
or equivalently
\begin{equation*}
\boxed{ 0.01 \geq N(0.4)(0.6)^{N-1} + (0.6)^N }
\end{equation*}
as our analytical expression for $N$.

In [24]:
# Solving for the smallest N numerically
def N_func(N):
    return 1 - N*0.4*0.6**(N-1) - 0.6**N

N = 0 
while N_func(N) < 0.99:                     # loop while condition is not met (N value does not give 0.99 or higher)
    print(f"N={N}: P={N_func(N)}")
    N+=1                                    # check next N
print(f"\nThe smallest N that gives a probability of 99% or higher is N={N}: P={N_func(N)}")       # print smallest N

N=0: P=0.0
N=1: P=0.0
N=2: P=0.16000000000000003
N=3: P=0.352
N=4: P=0.5248000000000002
N=5: P=0.6630400000000001
N=6: P=0.76672
N=7: P=0.8413696
N=8: P=0.89362432
N=9: P=0.929456128
N=10: P=0.9536425984
N=11: P=0.9697669120000001
N=12: P=0.980408958976
N=13: P=0.9873746624512

The smallest N that gives a probability of 99% or higher is N=14: P=0.99190236971008


Therefore, the smallest sample of $N$ catalogue objects should be $N=14$ object in order to ensure that the probability that at least two sample objects are non-binary stars is at least 99\%.

## b) 

Write Python code to simulate the scenario and test your solution and demonstrate that it is correct.

In [38]:
rng = np.random.default_rng()
n_trials = 1000000                                              # Repeat the experiment a million times -> draw N stars n_trials times     
N = 14                                                          # Draw 14 stars

In [53]:
non_binary = rng.random((n_trials, N)) < 0.4                    # Generates a table of uniformly distributed random numbers between 0 and 1, check if less than 0.4
counts = non_binary.sum(axis=1)                                 # Sum how many are 0.4 (counts how many 1s in the array of 1s and 0s)
counts                                                          # array of how many NB in that trial, one value per each of the million trials

array([5, 5, 7, ..., 6, 6, 2], shape=(1000000,))

In [59]:
successes_mask = (counts >= 2)                                          # Mask of where X >= 2
successes = np.count_nonzero(successes_mask)                            # How many successes
simulated_probability = successes / n_trials                            # Prob = successes / N
simulated_probability

np.float64(0.992034)

In [ ]:
theoretical_probability = N_func(N)
theoretical_probability

0.99190236971008

In [ ]:
stderr = np.sqrt(simulated_probability * (1 - simulated_probability) / n_trials)                        # Bernoulli Standard error
print(f"N = {N}: simulated probability = {simulated_probability:.5f} ± {stderr:.5f},  theoretical probability = {theoretical_probability:.5f}")

N = 14: simulated probability = 0.99203 ± 0.00009,  theoretical probability = 0.99190


In [64]:
# case N = 13
rng = np.random.default_rng()
n_trials = 1000000                                              # Repeat the experiment a million times -> draw N stars n_trials times     
N = 13                                                          # Draw 13 stars
non_binary = rng.random((n_trials, N)) < 0.4                    # Generates a table of uniformly distributed random numbers between 0 and 1, check if less than 0.4
counts = non_binary.sum(axis=1)                                 # Sum how many are 0.4 (counts how many 1s in the array of 1s and 0s)
successes_mask = (counts >= 2)                                          # Mask of where X >= 2
successes = np.count_nonzero(successes_mask)                            # How many successes
simulated_probability = successes / n_trials                            # Prob = successes / N
theoretical_probability = N_func(N)
stderr = np.sqrt(simulated_probability * (1 - simulated_probability) / n_trials)                        # Bernoulli Standard error
print(f"N = {N}: simulated probability = {simulated_probability:.5f} ± {stderr:.5f},  theoretical probability = {theoretical_probability:.5f}")


N = 13: simulated probability = 0.98721 ± 0.00011,  theoretical probability = 0.98737


# Q3
A galaxy of absolute magnitude $M = − 20$ is observed to have apparent magnitude
$m = 20.0\pm 0.2$. 

What is the luminosity distance $D_L$ in Mpc, and its error? [Assume $m− M = 5 \log_{10} D_L + 25$]

## a)
Solve analytically (LaTeX math).

## b)
Check your analytic solution with a Python simulation (code). Compare and discuss

How large does the uncertainty in $m$ have to be before your analytic solution for the error over/underestimates the true error by 10%. 

# Q5

See the following two data sets, which contain recession velocities and distances from Lemaitre (1927) and Hubble (1929).


The object of this exercise is to establish "who discovered the distance-redshift relation".

For each dataset, determine:

---

1. The Pearson product-moment correlation coefficient and its error, and the statistical significance of the correlation.

2. The Spearman rank cross-correlation coefficient and its statistical significance.

---

3. Linear least-squares regression lines of the form:
$$
V = HD
$$

    1. Starting from the notes or text, _derive_ an equation from first prinmciples for the best-fit slope $H$ and apply your equation to the data (i.e. do not use a built in fitting routine)

    2. Assuming that all the scatter is due to measurement errors in the velocity $V$, estimate these measurement errors (you may assume they are homoskedastic).

    3. Given the above, derive an expression for the uncertainty in the slope and apply it to the data. Apply your equation.

    4. Print the $H$, its uncertainty and the scatter around the fit.

    5. Plot the best fit as line with a shaded band indicating the $\pm1 \sigma$ uncertainties.

---

4. Now assume that the uncertainty is all in the measurements of $D$. Fit 

$$
D  = \frac{V}{H^{\sim}}
$$

Print $H^{\sim}$, its uncertainty and the scatter in $D$ around the best fit. Plot this on the same diagram as the previous line. 

Does $H^{\sim} = H$? Why or why not? If not, which one do you think is correct?

---

5. Linear least-squares regression lines of the form:
$$
V = H' D + C \; .
$$

You can fit this with a standard piece of code (e.g. polyfit)

Plot the best fit line and a shaded shape showing the uncertainty in the predicted value of the dependent variable over the entitre range spanned by the independent variable. Note this shape will not be linear!

Where is this shaded region thinnest? Is there anything special about this location?

Print the slope and intercept with uncertainties, and the scatter around the fit.

---

Who do you believe should be credited with the discovery of a linear redshift-distance relation? Justify your answer.


In [26]:
# Put the data into astropy tables

lemaitre_data_str = """# Lemaitre 1927 data
# Distance D is in Mpc and Velocity V is in km/s
# D   V
0.63 -300
0.11 -315
2.75  650
1.82  -25
1.66 1800
0.28  -70
1.82 1300
1.20  300
0.72  916
3.47  800
1.51  700
1.05  400
0.83  600
0.50  -30
0.69  290
0.87  600
1.10  940
0.83  812
1.91  600
1.15  730
1.05  800
0.72  650
1.15  800
2.75  950
2.00  300
0.60  500
1.10  500
0.87  200
0.63  850
0.95  800
1.82  580
1.74 1100
0.72 1140
0.87 1090
0.52  290
0.69  150
1.82  900
0.91  450
0.33  240
1.32  500
2.40  650
1.32  500
"""

import astropy.table

# read from the string as if it were a file
lemaitre = astropy.table.Table.read(lemaitre_data_str,
                                    format='ascii.commented_header',
                                    header_start=-1)

hubble_data_str="""# Hubble's 1929 data
# Distance D is in Mpc and Velocity V is in km/s 
#D     V
0.032  170
0.03   290
0.214 -130
0.263  -70
0.275 -185
0.275 -220
0.45   200
0.5    290
0.5    270
0.63   200
0.8    300
0.9    -30
0.9    650
0.9    150
0.9    500
1.0    920
1.1    450
1.1    500
1.4    500
1.7    960
2.0    500
2.0    850
2.0    800
2.0   1090
"""

hubble = astropy.table.Table.read(hubble_data_str,
                                  format='ascii.commented_header',
                                  header_start=-1)

In [27]:
hubble # print it to illustrate the data

D,V
float64,int64
0.032,170
0.03,290
0.214,-130
0.263,-70
0.275,-185
0.275,-220
0.45,200
0.5,290
0.5,270
